# Planning Pattern - Plan, Observe, Replan


This guided notebook shows the core Planning Pattern: build an initial timeline plan from some case artifact files, inspect a new record, and revise the plan when it changes what the evidence can support.

First, you make the planning steps visible manually. Then you compare them with `PlanningAgent`. This notebook does not use a separate `ReactAgent`; `03b` and `03c` introduce that workflow.

## How This Guided Exercise Works

The notebook code controls the teaching sequence:

1. **Initial Plan:** The notebook intentionally gives the LLM only selected case artifact files and asks it to create an initial plan. The network-status log is held back.
2. **New Observation:** The notebook supplies the network-status log after the initial plan.
3. **Revised Plan:** The LLM uses that new observation to revise its earlier plan.

**Purpose:** Compare the initial and revised plans to observe whether new evidence causes the LLM to update its earlier assumptions and next steps.

The LLM does not discover or reveal the held-back log by itself; it only works with the files the notebook gives it. Your job is to judge whether the revised plan responds appropriately to the new information.

![Figure 1. Manual planning and replanning workflow](figures/03a_manual_planning_workflow.svg)

*Figure 1. The core `03a` workflow: begin with the files currently supplied by the notebook, make an initial plan, compare a new observation with that plan, revise it when necessary, and write an evidence-bounded timeline. If a key gap remains, repeat the observation-and-replan steps.*

Your goal is a final timeline that states what the records support and what remains unknown.


## Lab Question and What to Record

**What timeline would you build from the partial artifact bundle, and how should that timeline change when the network-status artifact is added later in the full artifact bundle?**

Record these four items as you work:

1. **Initial investigation plan:** goal, ordered steps, needed evidence, and replanning triggers.
2. **Observation-driven replan:** what changed and how the remaining steps change.
3. **Reconstructed timeline and timing conclusion:** key events in order and their timing.
4. **Evidence mapping and remaining uncertainty:** supporting artifacts and limits.

Run the notebook from top to bottom. You do not need to write new Python; focus on why the new observation changes the plan.


## Notebook Structure

- **Shared setup (Steps 1–2):** Load the lab settings and create the case-record bundles used by both parts.
- **Part A — Manual Planning (Steps 3–5):** Make the planning process visible: create an initial plan, receive the new observation, revise the plan, and write a bounded conclusion.
- **Part B — `PlanningAgent` (Steps 6–9):** Run the same process through the packaged agent and compare its results with Part A.

Run the shared setup once before completing Part A and Part B. Both parts use the same staged case records; the difference is whether you see each planning step directly or through `PlanningAgent`.


## Shared Setup

### Step 1: Set Up the Notebook


Run this cell first. It loads the lab settings, checks the notebook location, connects to the model, and prepares the case data.


In [ ]:
# Purpose: This cell supports Step 1: Set Up the Notebook by loading the libraries, settings, and data needed for this section.
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and prepares the case data.
LAB_NAME = 'lab4_planning_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError('Could not find the Lab 4 data folder')

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)


### Step 2: Build the Partial and Full Artifact Bundles

The initial plan uses `partial_artifact_bundle`, which intentionally omits the network record. Later, `new_observation` supplies that record and requires a replan. `full_artifact_bundle` combines both for the final conclusion.


In [ ]:
# Purpose: This cell supports Step 2: Build the Partial and Full Artifact Bundles by defining reusable helper code that performs the work described here.
# Show only a short excerpt from each file so the notebook stays readable.
def excerpt(filename: str, n_lines: int = 8) -> str:
    lines = (data_dir / filename).read_text().strip().splitlines()
    return '\n'.join(lines[:n_lines])


# The initial bundle intentionally leaves out network_status.csv.
partial_artifact_bundle = f"""
artifact_manifest.json
{(data_dir / 'artifact_manifest.json').read_text()}

unlock_events.csv
{excerpt('unlock_events.csv')}

call_log.csv
{excerpt('call_log.csv')}

whatsapp_events.csv
{excerpt('whatsapp_events.csv')}

chain_of_custody.csv
{excerpt('chain_of_custody.csv')}
"""

# This simulates a newly discovered observation that arrives after the initial plan.
new_observation = f"""
network_status.csv
{excerpt('network_status.csv')}
"""

# The full bundle is used only after the plan has been revised.
full_artifact_bundle = partial_artifact_bundle + '\n\n' + new_observation


## Part A: Walk the Planning Workflow Manually

Build an initial plan from the available records, inspect the new observation, revise the plan, and then write a bounded timeline conclusion.


### Step 3: Build the Initial Investigation Plan

This step asks the model to create an ordered investigation path from the partial artifact bundle. Because the network-status record is still hidden, the initial plan should include possible replanning triggers instead of pretending the first plan is complete.


In [ ]:
# Purpose: This cell supports Step 3: Build the Initial Investigation Plan by running the model or agent action and saving its result for review.
PLANNING_SYSTEM_PROMPT = """
You are a digital forensics planning assistant.
Build ordered investigation plans, update them when new observations expose missing dependencies, and avoid unsupported conclusions.
"""

case_question = (
    'What timeline would you build from the partial artifact bundle, ' 
    'and how should that timeline change when the network-status artifact is added later in the full artifact bundle?'
)

# Ask the model for an initial plan using only the partial artifact bundle.
# `replanning triggers` means: what future evidence or contradiction should make the current plan change.
initial_plan_request = (
    'Build an initial investigation plan for this case.\n'
    'Return a plan with: (1) investigation goal, (2) ordered steps, (3) evidence needed, ' 
    '(4) replanning triggers.\n\n'
    f'Task:\n{case_question}\n\n'
    f'Artifacts:\n{partial_artifact_bundle}'
)

# Run the model once to get the initial plan.
initial_plan = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': PLANNING_SYSTEM_PROMPT},
        {'role': 'user', 'content': initial_plan_request},
    ],
    model=MODEL,
).choices[0].message.content

display(Markdown('### Initial Investigation Plan\n\n' + initial_plan))


### Step 4: Revise the Plan After the New Observation

The network record changes what the earlier WhatsApp activity can support. Update the plan instead of continuing with the original assumptions.


In [ ]:
# Purpose: This cell supports Step 4: Revise the Plan After the New Observation by running the model or agent action and saving its result for review.
# Use the new observation to update the current plan rather than starting over from scratch.
# `remaining replanning triggers` means: what future finding would still require another revision after this replan.
revised_plan_request = (
    'Revise the current investigation plan using the new observation below.\n'
    'Return a revised plan with: (1) what changed, (2) revised ordered steps, (3) updated evidence priorities, ' 
    '(4) remaining replanning triggers.\n\n'
    f'Original task:\n{case_question}\n\n'
    f'Current plan:\n{initial_plan}\n\n'
    f'New observation:\n{new_observation}'
)

revised_plan = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': PLANNING_SYSTEM_PROMPT},
        {'role': 'user', 'content': revised_plan_request},
    ],
    model=MODEL,
).choices[0].message.content

display(Markdown('### Observation-Driven Replan\n\n' + revised_plan))


### Step 5: Write the Final Timeline Conclusion

Use the revised plan and full artifact bundle to write a bounded report. Separate in-window activity from anything that may have happened after reconnection.


In [ ]:
# Purpose: This cell supports Step 5: Write the Final Timeline Conclusion by running the model or agent action and saving its result for review.
# Ask for a final report only after the initial plan has been updated with the new observation.
final_report_request = (
    'Using the revised plan and full artifact set, produce a short evidence-cited timeline conclusion.\n'
    'Return a report with: (1) reconstructed timeline, (2) timing conclusion, (3) evidence mapping, ' 
    '(4) what remains uncertain.\n\n'
    f'Revised plan:\n{revised_plan}\n\n'
    f'Artifacts:\n{full_artifact_bundle}'
)

final_timeline_conclusion = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': PLANNING_SYSTEM_PROMPT},
        {'role': 'user', 'content': final_report_request},
    ],
    model=MODEL,
).choices[0].message.content

display(Markdown('### Reconstructed Timeline and Timing Conclusion\n\n' + final_timeline_conclusion))


## Part B: Run the Same Workflow with `PlanningAgent`

`PlanningAgent` packages the same workflow: build a plan, revise it with a new observation, and produce a final report. Compare it with Part A rather than assuming it is automatically better.


### Step 6: Create the `PlanningAgent`

This cell creates the packaged agent for the workflow you just completed manually.


In [ ]:
# Purpose: This cell supports Step 6: Create the `PlanningAgent` by loading the libraries, settings, and data needed for this section.
from agentic_patterns.planning_pattern.planning_agent import PlanningAgent

planning_agent = PlanningAgent(client=client, model=MODEL)


### Step 7: Let `PlanningAgent` Build the Initial Plan

This run uses the same partial bundle as Part A. Compare its plan with the manual plan.


In [ ]:
# Purpose: This cell supports Step 7: Let `PlanningAgent` Build the Initial Plan by preparing or examining the evidence used in this section.
packaged_initial_plan = planning_agent.build_initial_plan(
    f'{case_question}\n\nArtifacts:\n{partial_artifact_bundle}'
)
display(Markdown('### PlanningAgent Initial Plan\n\n' + packaged_initial_plan))


### Step 8: Let `PlanningAgent` Revise the Plan

The new network observation asks the agent to update its plan. Check whether its revision is evidence-based.


In [ ]:
# Purpose: This cell supports Step 8: Let `PlanningAgent` Revise the Plan by preparing or examining the evidence used in this section.
packaged_revised_plan = planning_agent.revise_plan(
    user_msg=f'{case_question}\n\nArtifacts:\n{partial_artifact_bundle}',
    current_plan=packaged_initial_plan,
    observations=new_observation,
)
display(Markdown('### PlanningAgent Revised Plan\n\n' + packaged_revised_plan))


### Step 9: Run the Full Planning Workflow

Compare the agent's final report with your manual conclusion. Does it preserve the same evidence limits?


In [ ]:
# Purpose: This cell supports Step 9: Run the Full Planning Workflow by running the model or agent action and saving its result for review.
packaged_final_report = planning_agent.run(user_msg=f'{case_question}\n\nArtifacts:\n{full_artifact_bundle}')
display(Markdown('### PlanningAgent Final Report\n\n' + packaged_final_report))


## Key Takeaway

Part A makes planning visible: plan -> observe -> replan -> conclude. Part B packages the same steps in `PlanningAgent`. In both versions, revise the path when new evidence changes what the earlier plan can support.

**Limitation of this notebook:** The Python code manually supplies the new observation. It does not choose the next evidence to inspect.

In `03b_guided_planner_react_workflow.ipynb`, `PlanningAgent` chooses the next evidence-gathering task and `ReactAgent` uses approved tools to carry it out. The returned observation lets the planner revise the plan. The program still limits the agents to approved tools and case files.
